# Cleaning and Multi-File Integration

## Objective

Build the canonical cleaned dataset from the approved nine-source mapping while preserving traceability for every rejected row.

## Questions being answered

- Are source columns normalized to the canonical snake_case schema?
- Which rows, if any, fail explicit validity rules?
- Does the raw-included minus rejected-row reconciliation balance?
- Are zero-tax records retained?
- Are legacy files excluded from the canonical output?

## Imports

In [ ]:
import pandas as pd

from src.cleaning import build_clean_dataset, write_clean_outputs
from src.config import RAW_CANONICAL_COLUMNS

## Data-loading section

`build_clean_dataset` loads only the explicit canonical source mapping from `src.config`. It does not use `glob`, and it does not edit files under `data/raw/`.

In [ ]:
cleaned, rejected, cleaning_summary = build_clean_dataset()
output_paths = write_clean_outputs(cleaned, rejected, cleaning_summary)
print(f"Cleaned rows: {len(cleaned):,}")
print(f"Rejected rows: {len(rejected):,}")
print(output_paths)

## Methodology

The pipeline adds the explicit brand and original source filename, renames source-specific columns, trims only relevant strings, converts numeric values with a conversion-failure log, and applies documented validity rules. It rejects only invalid rows. It does not remove duplicates because the source has no listing identifier and no approved duplicate-removal rule yet.

## Results: canonical schema

In [ ]:
print(cleaned.columns.tolist())
cleaned.head()

## Results: row reconciliation

In [ ]:
cleaning_summary

## Results: rejected rows

Rejected rows are written to `data/interim/rejected_rows.csv` with source filename, one-based source row number, normalized source values, and a rejection reason.

In [ ]:
rejected

## Validation checks

The canonical output uses the stable raw canonical column order and retains valid `tax == 0` records. The legacy model-only files are not loaded.

In [ ]:
assert cleaned.columns.tolist() == list(RAW_CANONICAL_COLUMNS)
assert cleaning_summary["reconciliation_status"].eq("balanced").all()
assert (cleaned["tax"] == 0).any()
print("All cleaning checks passed.")

## Interpretation

The cleaned dataset is suitable for later analysis only after the rejected-row count and reasons are reviewed. A balanced reconciliation shows that every source row is accounted for as either accepted or rejected; it does not mean every accepted row is unique.

## Data-quality or analytical limitations

The validation rules are based on the Phase 2 audit and dataset conventions. The year lower bound is documented as 1970, while the fixed upper bound is 2024. Zero engine size and duplicate-looking listings remain visible for later investigation.

## Findings and Decisions

- Use the nine complete manufacturer-level files only.
- Keep raw files immutable.
- Reject only rows failing explicit rules, with traceable reasons.
- Preserve zero-tax rows.
- Do not deduplicate until a separate policy is approved.